# Federated streaming detection: results analysis

Core narrative: a shared static novelty baseline (frozen scoring model +
bootstrap-calibrated threshold) produces consistent, domain-sensitive selection
behavior across heterogeneous federated clients. Clients in novel environments
naturally accept more frames, while clients in the familiar domain are selective.
Two bootstrap compositions are compared: city-day (narrow) and city-mixed (diverse).

The focus is on characterizing per-client selection behavior and comparing
federated outcomes with centralized streaming under matched filter policies.

Sections:

1. **Setup** -- run discovery, client partitioning, bootstrap vs stream composition.
2. **Global mAP per round** -- detection performance convergence.
3. **Per-class AP per round** -- Pedestrian AP as domain-shift indicator.
4. **Per-client accept rates** -- how each client's filter responds to its domain.
5. **Per-client training effort** -- items processed, accepted, optimizer steps.
6. **Bandwidth efficiency** -- mAP vs total frames sent.
7. **Comparison with streaming** -- federated vs centralized on same manifest.
8. **Summary table** -- final metrics for all variants.

See `01_streaming_analysis.ipynb` for streaming (centralized) results.

## 1 Setup

In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

%matplotlib inline
plt.rcParams.update({"figure.dpi": 140, "font.size": 9,
                     "axes.titlesize": 10, "axes.labelsize": 9,
                     "legend.fontsize": 8, "xtick.labelsize": 8,
                     "ytick.labelsize": 8, "figure.facecolor": "white"})

sys.path.insert(0, str(Path.cwd() / "notebooks"))
if not (Path.cwd() / "pyproject.toml").exists():
    sys.path.insert(0, str(Path.cwd().parent / "notebooks"))

import analysis_helpers as ah

PROJECT_ROOT = ah.find_project_root()
OUTPUTS = PROJECT_ROOT / "outputs"

SEED: int | None = None

PALETTE: dict[str, str] = {
    # City-day bootstrap
    "fed_no_filter_cityday_road_type": "#2ca02c",
    "fed_random_filter_cityday_road_type": "#ff7f0e",
    "fed_dist_thresh_cityday_road_type_p10": "#1f77b4",
    "fed_dist_thresh_cityday_reverse_p10": "#d62728",
    # City-mixed bootstrap
    "fed_no_filter_citymix_road_type": "#8c564b",
    "fed_random_filter_citymix_road_type": "#bcbd22",
    "fed_dist_thresh_citymix_road_type_p10": "#9467bd",
    "fed_dist_thresh_citymix_reverse_p10": "#e377c2",
    "fed_dist_thresh_citymix_conditions_p10": "#17becf",
}

SHORT_NAMES: dict[str, str] = {
    # City-day bootstrap
    "fed_no_filter_cityday_road_type": "No filter (city-day)",
    "fed_random_filter_cityday_road_type": "Random 55% (city-day)",
    "fed_dist_thresh_cityday_road_type_p10": "Dist-filter road_type (city-day)",
    "fed_dist_thresh_cityday_reverse_p10": "Dist-filter reverse (city-day)",
    # City-mixed bootstrap
    "fed_no_filter_citymix_road_type": "No filter (city-mix)",
    "fed_random_filter_citymix_road_type": "Random 55% (city-mix)",
    "fed_dist_thresh_citymix_road_type_p10": "Dist-filter road_type (city-mix)",
    "fed_dist_thresh_citymix_reverse_p10": "Dist-filter reverse (city-mix)",
    "fed_dist_thresh_citymix_conditions_p10": "Dist-filter conditions (city-mix)",
}

print("Project:", PROJECT_ROOT)

In [ ]:
runs_df = ah.discover_runs(OUTPUTS)


def pick(pipeline: str, variant: str, seed: int | None = SEED) -> Path | None:
    return ah.pick_latest_run(runs_df, pipeline, variant, seed=seed)


def _runs_non_null(raw: dict[str, Path | None]) -> dict[str, Path]:
    out: dict[str, Path] = {}
    for k, v in raw.items():
        if v is not None:
            out[k] = v
    return out


RUN: dict[str, Path] = _runs_non_null(
    {
        # City-day bootstrap
        "fed_no_filter_cityday_road_type": pick("federated", "fed_no_filter_cityday_road_type"),
        "fed_random_filter_cityday_road_type": pick("federated", "fed_random_filter_cityday_road_type"),
        "fed_dist_thresh_cityday_road_type_p10": pick("federated", "fed_dist_thresh_cityday_road_type_p10"),
        "fed_dist_thresh_cityday_reverse_p10": pick("federated", "fed_dist_thresh_cityday_reverse_p10"),
        # City-mixed bootstrap
        "fed_no_filter_citymix_road_type": pick("federated", "fed_no_filter_citymix_road_type"),
        "fed_random_filter_citymix_road_type": pick("federated", "fed_random_filter_citymix_road_type"),
        "fed_dist_thresh_citymix_road_type_p10": pick("federated", "fed_dist_thresh_citymix_road_type_p10"),
        "fed_dist_thresh_citymix_reverse_p10": pick("federated", "fed_dist_thresh_citymix_reverse_p10"),
        "fed_dist_thresh_citymix_conditions_p10": pick("federated", "fed_dist_thresh_citymix_conditions_p10"),
    }
)

# Load rounds.csv for each run
ROUNDS: dict[str, pd.DataFrame] = {}
for k, p in sorted(RUN.items()):
    rd = ah.read_csv(p / "rounds.csv")
    if rd is not None and not rd.empty:
        ROUNDS[k] = rd
        cfg = ah.load_run_config(p)
        print(f"{k:50s}  rounds={len(rd)}  seed={cfg.get('seed')}")
        print(f"{'':50s}  {p}")
    else:
        print(f"{k:50s}  [no rounds.csv]")

print(f"\nLoaded {len(ROUNDS)} federated runs.")

In [ ]:
# --- Client partitioning summary ---
_bootstrap_frames = 5000

for k, p in sorted(RUN.items()):
    cfg = ah.load_run_config(p)
    man = ah.load_manifest(PROJECT_ROOT, str(cfg.get("manifest_path", "")))
    if not man:
        continue
    n_train = sum(1 for f in man.get("frames", []) if f.get("split") == "train")
    stream = n_train - _bootstrap_frames
    n_clients = int(cfg.get("num_clients", 4))
    per_client = stream // n_clients

    ordering = man.get("ordering", {})
    block_order = ordering.get("block_order", [])
    block_sizes = ordering.get("block_sizes", {})

    print(f"\n{k}")
    print(f"  Stream: {stream} frames, {n_clients} clients, ~{per_client}/client")
    print(f"  Blocks: {block_order}")

    # Show which blocks each client gets
    pos = 0
    client_start = 0
    for cid in range(n_clients):
        base = stream // n_clients
        extra = 1 if cid < (stream % n_clients) else 0
        client_end = client_start + base + extra
        # Find which blocks overlap this client's range
        block_pos = 0
        domains = []
        for blk in block_order:
            bsz = block_sizes.get(blk, 0)
            blk_start = block_pos
            blk_end = block_pos + bsz
            if blk_end > client_start and blk_start < client_end:
                overlap = min(blk_end, client_end) - max(blk_start, client_start)
                domains.append(f"{blk}({overlap})")
            block_pos += bsz
        print(f"  Client {cid}: [{client_start}, {client_end})  domains: {', '.join(domains)}")
        client_start = client_end

### Client domain composition

What each client actually sees: road types, time of day, weather, and
Pedestrian density per client partition. Derived from the manifest metadata -- no
experiment output needed.

In [ ]:
from collections import Counter
from matplotlib.patches import Patch
import matplotlib.colors as mcolors

_DOMAIN_COLORS: dict[str, str] = {
    "city": "#1f77b4",
    "arterial-urban": "#ff7f0e",
    "highway": "#2ca02c",
    "arterial-rural": "#d62728",
    "smaller-rural": "#9467bd",
}
_TOD_COLORS: dict[str, str] = {
    "day": "#f7dc6f",
    "dawn/dusk": "#e59866",
    "night": "#2c3e50",
}


def _client_partitions(
    manifest: dict, n_clients: int, bootstrap_frames: int = 5000,
) -> dict[int, pd.DataFrame]:
    """Return a DataFrame of stream frames per client, with full metadata."""
    mdf = ah.manifest_to_dataframe(manifest)
    train_mask = mdf["split"] == "train"
    train_df = mdf.loc[train_mask].reset_index(drop=True)
    stream = train_df.iloc[bootstrap_frames:]
    n = len(stream)
    partitions: dict[int, pd.DataFrame] = {}
    start = 0
    for cid in range(n_clients):
        base = n // n_clients
        extra = 1 if cid < (n % n_clients) else 0
        end = start + base + extra
        partitions[cid] = stream.iloc[start:end].reset_index(drop=True)
        start = end
    return partitions


def plot_client_domain_bars(
    manifest: dict,
    n_clients: int,
    bootstrap_frames: int = 5000,
    title: str = "",
    field: str = "road_type",
    color_map: dict[str, str] | None = None,
) -> None:
    """Stacked horizontal bar: domain composition per client."""
    parts = _client_partitions(manifest, n_clients, bootstrap_frames)
    if color_map is None:
        color_map = _DOMAIN_COLORS if field == "road_type" else _TOD_COLORS
    all_cats = []
    for cid in sorted(parts):
        for v in parts[cid][field].unique():
            if v not in all_cats:
                all_cats.append(v)

    fig, ax = plt.subplots(figsize=(10, 0.6 * n_clients + 1.2))
    for cid in sorted(parts):
        counts = Counter(parts[cid][field])
        total = sum(counts.values())
        left = 0.0
        for cat in all_cats:
            w = counts.get(cat, 0) / total
            ax.barh(cid, w, left=left, color=color_map.get(cat, "#aaaaaa"),
                    edgecolor="white", linewidth=0.3)
            if w > 0.08:
                ax.text(left + w / 2, cid, f"{w:.0%}", ha="center", va="center",
                        fontsize=7, color="white" if cat in ("night",) else "black")
            left += w
    ax.set_yticks(range(n_clients))
    ax.set_yticklabels([f"Client {i}" for i in range(n_clients)])
    ax.set_xlabel("Fraction of frames")
    ax.set_title(title or f"Client {field} composition")
    ax.legend(
        handles=[Patch(facecolor=color_map.get(c, "#aaa"), label=c) for c in all_cats],
        loc="upper right", fontsize=7, ncol=min(len(all_cats), 3),
    )
    ax.invert_yaxis()
    plt.tight_layout()
    plt.show()


def plot_client_stream_stripes(
    manifest: dict,
    n_clients: int,
    bootstrap_frames: int = 5000,
    title: str = "",
    field: str = "road_type",
    color_map: dict[str, str] | None = None,
    downsample: int = 20,
) -> None:
    """Stripe plot: each row is a client, each pixel-column is a frame colored by domain.

    `downsample` controls the resolution (1 column per N frames).
    """
    parts = _client_partitions(manifest, n_clients, bootstrap_frames)
    if color_map is None:
        color_map = _DOMAIN_COLORS if field == "road_type" else _TOD_COLORS
    all_cats = list(color_map.keys())
    cat_to_idx = {c: i for i, c in enumerate(all_cats)}
    cmap = mcolors.ListedColormap([color_map.get(c, "#aaa") for c in all_cats])
    norm = mcolors.BoundaryNorm(range(len(all_cats) + 1), cmap.N)

    max_len = max(len(p) for p in parts.values())
    cols = (max_len + downsample - 1) // downsample
    img = np.full((n_clients, cols), np.nan)
    for cid in sorted(parts):
        vals = parts[cid][field].values
        for j in range(0, len(vals), downsample):
            chunk = vals[j : j + downsample]
            most_common = Counter(chunk).most_common(1)[0][0]
            img[cid, j // downsample] = cat_to_idx.get(most_common, np.nan)

    fig, ax = plt.subplots(figsize=(12, 0.8 * n_clients + 1.0))
    ax.imshow(img, aspect="auto", cmap=cmap, norm=norm, interpolation="nearest")
    ax.set_yticks(range(n_clients))
    ax.set_yticklabels([f"Client {i}" for i in range(n_clients)])
    xticks_pos = np.linspace(0, cols - 1, 6).astype(int)
    ax.set_xticks(xticks_pos)
    ax.set_xticklabels([f"{int(x * downsample):,}" for x in xticks_pos])
    ax.set_xlabel("Frame index within client partition")
    ax.set_title(title or f"Client stream: {field}")
    ax.legend(
        handles=[Patch(facecolor=color_map.get(c, "#aaa"), label=c) for c in all_cats],
        loc="upper right", fontsize=7, ncol=min(len(all_cats), 3),
        bbox_to_anchor=(1.0, -0.08),
    )
    plt.tight_layout()
    plt.show()


def print_client_metadata_summary(
    manifest: dict,
    n_clients: int,
    bootstrap_frames: int = 5000,
) -> None:
    """Print a compact summary of each client's metadata: ToD, weather, Pedestrian density."""
    parts = _client_partitions(manifest, n_clients, bootstrap_frames)
    rows = []
    for cid in sorted(parts):
        df = parts[cid]
        n = len(df)
        tod = Counter(df["time_of_day"])
        ped_frac = (df["num_pedestrians"] > 0).sum() / n if n else 0
        avg_ped = df["num_pedestrians"].mean()
        weather_top3 = Counter(df["scraped_weather"]).most_common(3)
        rows.append({
            "client": cid,
            "frames": n,
            "day%": f"{tod.get('day', 0)/n:.0%}",
            "dawn/dusk%": f"{tod.get('dawn/dusk', 0)/n:.0%}",
            "night%": f"{tod.get('night', 0)/n:.0%}",
            "ped_presence": f"{ped_frac:.0%}",
            "avg_ped": f"{avg_ped:.1f}",
            "top_weather": ", ".join(f"{w}({c/n:.0%})" for w, c in weather_top3),
        })
    display(pd.DataFrame(rows).set_index("client"))


# --- Generate for each loaded run ---
for k, p in sorted(RUN.items()):
    cfg = ah.load_run_config(p)
    man = ah.load_manifest(PROJECT_ROOT, str(cfg.get("manifest_path", "")))
    if not man:
        print(f"[skip] {k}: no manifest")
        continue
    n_clients = int(cfg.get("num_clients", 4))
    short = SHORT_NAMES.get(k, k)

    print(f"\n{'='*70}")
    print(f"  {short}  ({k})")
    print(f"{'='*70}")

    plot_client_domain_bars(man, n_clients, title=f"{short} -- road type per client")
    plot_client_domain_bars(man, n_clients, field="time_of_day",
                           color_map=_TOD_COLORS,
                           title=f"{short} -- time of day per client")
    plot_client_stream_stripes(man, n_clients, title=f"{short} -- stream domains (road type)")
    plot_client_stream_stripes(man, n_clients, field="time_of_day",
                               color_map=_TOD_COLORS,
                               title=f"{short} -- stream domains (time of day)")
    print_client_metadata_summary(man, n_clients)
    print()

### Bootstrap vs. stream domain distribution

Mismatch between what the initial model was trained on (bootstrap)
and what the stream contains. This is the source of novelty that the distribution
filter should detect.

In [ ]:
def plot_bootstrap_vs_stream(
    manifest: dict,
    bootstrap_frames: int = 5000,
    title: str = "",
) -> None:
    """Side-by-side bar charts comparing bootstrap and stream compositions."""
    mdf = ah.manifest_to_dataframe(manifest)
    train = mdf.loc[mdf["split"] == "train"].reset_index(drop=True)
    boot = train.iloc[:bootstrap_frames]
    stream = train.iloc[bootstrap_frames:]

    fields = ["road_type", "time_of_day"]
    fig, axes = plt.subplots(1, len(fields), figsize=(12, 3.5))
    for ax, field in zip(axes, fields):
        cmap = _DOMAIN_COLORS if field == "road_type" else _TOD_COLORS
        boot_counts = Counter(boot[field])
        stream_counts = Counter(stream[field])
        all_cats = sorted(set(list(boot_counts) + list(stream_counts)),
                          key=lambda c: -(boot_counts.get(c, 0) + stream_counts.get(c, 0)))
        x = np.arange(len(all_cats))
        n_boot = len(boot)
        n_stream = len(stream)
        boot_frac = [boot_counts.get(c, 0) / n_boot for c in all_cats]
        stream_frac = [stream_counts.get(c, 0) / n_stream for c in all_cats]

        w = 0.35
        bars_b = ax.bar(x - w / 2, boot_frac, w, label="Bootstrap",
                        color=[cmap.get(c, "#aaa") for c in all_cats],
                        edgecolor="black", linewidth=0.8, alpha=0.6)
        bars_s = ax.bar(x + w / 2, stream_frac, w, label="Stream",
                        color=[cmap.get(c, "#aaa") for c in all_cats],
                        edgecolor="black", linewidth=0.8, alpha=1.0)
        ax.set_xticks(x)
        ax.set_xticklabels(all_cats, rotation=30, ha="right", fontsize=7)
        ax.set_ylabel("Fraction")
        ax.set_title(field.replace("_", " ").title())
        ax.legend(fontsize=7)
    fig.suptitle(title or "Bootstrap vs. stream composition", fontsize=10, y=1.02)
    plt.tight_layout()
    plt.show()


# --- Show for each run (grouped by unique manifest) ---
_seen_manifests: set[str] = set()
for k, p in sorted(RUN.items()):
    cfg = ah.load_run_config(p)
    mpath = str(cfg.get("manifest_path", ""))
    if mpath in _seen_manifests:
        continue
    _seen_manifests.add(mpath)
    man = ah.load_manifest(PROJECT_ROOT, mpath)
    if not man:
        continue
    short = SHORT_NAMES.get(k, k)
    ordering = man.get("ordering", {})
    boot_n = ordering.get("bootstrap_frames", 5000)
    print(f"\nManifest: {mpath}")
    print(f"  Strategy: {ordering.get('strategy', '?')}")
    print(f"  Bootstrap: {boot_n} frames")
    plot_bootstrap_vs_stream(man, bootstrap_frames=boot_n,
                             title=f"Bootstrap vs. stream -- {mpath.split('/')[-1]}")

## 2 Global mAP per round

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for k, rd in ROUNDS.items():
    c = PALETTE.get(k, "gray")
    label = SHORT_NAMES.get(k, k)
    if "mAP" in rd.columns:
        valid = rd[rd["mAP"].notna()]
        axes[0].plot(valid["round"].to_numpy(), valid["mAP"].astype(float).to_numpy(),
                     marker="o", ms=4, color=c, label=label)
    if "mAP_50" in rd.columns:
        valid = rd[rd["mAP_50"].notna()]
        axes[1].plot(valid["round"].to_numpy(), valid["mAP_50"].astype(float).to_numpy(),
                     marker="o", ms=4, color=c, label=label)

axes[0].set(xlabel="Round", ylabel="mAP", title="Global mAP per round")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].set(xlabel="Round", ylabel="mAP@50", title="Global mAP@50 per round")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

fig.tight_layout()
plt.show()

## 3 Per-class AP per round

In [ ]:
TARGET_CLASSES = ["Vehicle", "Pedestrian", "VulnerableVehicle"]

fig, axes = plt.subplots(1, len(TARGET_CLASSES), figsize=(5 * len(TARGET_CLASSES), 4))
if len(TARGET_CLASSES) == 1:
    axes = [axes]

for i, cls_name in enumerate(TARGET_CLASSES):
    col = f"AP_{cls_name}"
    ax = axes[i]
    for k, rd in ROUNDS.items():
        if col not in rd.columns:
            continue
        valid = rd[rd[col].notna()]
        if valid.empty:
            continue
        c = PALETTE.get(k, "gray")
        label = SHORT_NAMES.get(k, k)
        ax.plot(valid["round"].to_numpy(), valid[col].astype(float).to_numpy(),
                marker="o", ms=4, color=c, label=label)
    ax.set(xlabel="Round", ylabel=f"AP_{cls_name}", title=cls_name)
    ax.legend()
    ax.grid(True, alpha=0.3)

fig.tight_layout()
plt.show()

## 4 Per-client accept rates per round

In [ ]:
_CLIENT_COLORS = ["#1f77b4", "#ff7f0e", "#2ca02c", "#d62728",
                  "#9467bd", "#8c564b", "#e377c2", "#7f7f7f"]


def _client_domain_labels(
    manifest: dict, n_clients: int, bootstrap_frames: int = 5000,
    field: str = "road_type", top_n: int = 2,
) -> dict[int, str]:
    """Build short domain-annotated labels like 'Client 0 (city 100%)'."""
    parts = _client_partitions(manifest, n_clients, bootstrap_frames)
    labels: dict[int, str] = {}
    for cid in sorted(parts):
        counts = Counter(parts[cid][field])
        total = sum(counts.values())
        top = counts.most_common(top_n)
        desc = ", ".join(f"{c} {n/total:.0%}" for c, n in top)
        labels[cid] = f"Client {cid} ({desc})"
    return labels


dist_keys = [k for k in ROUNDS if "dist_thresh" in k]

for k in dist_keys:
    rd = ROUNDS[k]
    cfg = ah.load_run_config(RUN[k])
    n_clients = int(cfg.get("num_clients", 4))
    man = ah.load_manifest(PROJECT_ROOT, str(cfg.get("manifest_path", "")))
    if man:
        ordering = man.get("ordering", {})
        boot_n = ordering.get("bootstrap_frames", 5000)
        client_labels = _client_domain_labels(man, n_clients, boot_n)
    else:
        client_labels = {i: f"Client {i}" for i in range(n_clients)}

    fig, axes = plt.subplots(1, 2, figsize=(13, 4))
    fig.suptitle(SHORT_NAMES.get(k, k), fontsize=11)

    for cid in range(n_clients):
        items_col = f"client_{cid}_items"
        acc_col = f"client_{cid}_accepted"
        if items_col not in rd.columns or acc_col not in rd.columns:
            continue
        items = rd[items_col].astype(float)
        accepted = rd[acc_col].astype(float)
        rate = (accepted / items.replace(0, np.nan)).fillna(0)

        color = _CLIENT_COLORS[cid % len(_CLIENT_COLORS)]
        lbl = client_labels.get(cid, f"Client {cid}")
        axes[0].plot(rd["round"].to_numpy(), rate.to_numpy(),
                     marker="o", ms=3, color=color, label=lbl)
        axes[1].plot(rd["round"].to_numpy(), accepted.to_numpy(),
                     marker="o", ms=3, color=color, label=lbl)

    axes[0].set(xlabel="Round", ylabel="Accept rate", title="Per-client accept rate")
    axes[0].legend(fontsize=7)
    axes[0].grid(True, alpha=0.3)
    axes[0].set_ylim(-0.05, 1.05)

    axes[1].set(xlabel="Round", ylabel="Frames accepted", title="Per-client accepted frames")
    axes[1].legend(fontsize=7)
    axes[1].grid(True, alpha=0.3)

    fig.tight_layout()
    plt.show()

## 5 Per-client training effort

In [ ]:
for k in sorted(ROUNDS.keys()):
    rd = ROUNDS[k]
    cfg = ah.load_run_config(RUN[k])
    n_clients = int(cfg.get("num_clients", 4))
    man = ah.load_manifest(PROJECT_ROOT, str(cfg.get("manifest_path", "")))
    if man:
        ordering = man.get("ordering", {})
        boot_n = ordering.get("bootstrap_frames", 5000)
        clabels = _client_domain_labels(man, n_clients, boot_n)
    else:
        clabels = {i: f"Client {i}" for i in range(n_clients)}

    print(f"\n{SHORT_NAMES.get(k, k)}")
    effort_rows = []
    for cid in range(n_clients):
        items_col = f"client_{cid}_items"
        acc_col = f"client_{cid}_accepted"
        rej_col = f"client_{cid}_rejected"
        steps_col = f"client_{cid}_optimizer_steps"
        if items_col not in rd.columns:
            continue
        total_items = int(rd[items_col].astype(float).sum())
        total_acc = int(rd[acc_col].astype(float).sum()) if acc_col in rd.columns else 0
        total_rej = int(rd[rej_col].astype(float).sum()) if rej_col in rd.columns else 0
        total_steps = int(rd[steps_col].astype(float).sum()) if steps_col in rd.columns else 0
        rate = total_acc / total_items if total_items > 0 else 0
        effort_rows.append({
            "client": clabels.get(cid, f"Client {cid}"),
            "items": total_items,
            "accepted": total_acc,
            "accept_rate": f"{rate:.1%}",
            "rejected": total_rej,
            "opt_steps": total_steps,
        })
    display(pd.DataFrame(effort_rows).set_index("client"))

## 6 Bandwidth efficiency: mAP vs total frames sent

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

for k, rd in ROUNDS.items():
    cfg = ah.load_run_config(RUN[k])
    n_clients = int(cfg.get("num_clients", 4))
    c = PALETTE.get(k, "gray")
    label = SHORT_NAMES.get(k, k)

    # Cumulative total accepted across all clients
    cum_accepted = np.zeros(len(rd))
    for cid in range(n_clients):
        acc_col = f"client_{cid}_accepted"
        if acc_col in rd.columns:
            cum_accepted += rd[acc_col].astype(float).to_numpy()
    cum_accepted = np.cumsum(cum_accepted)

    if "mAP" in rd.columns:
        valid_mask = rd["mAP"].notna()
        ax.plot(cum_accepted[valid_mask.to_numpy()],
                rd.loc[valid_mask, "mAP"].astype(float).to_numpy(),
                marker="o", ms=4, color=c, label=label)

ax.set(xlabel="Cumulative frames accepted (all clients)",
       ylabel="Global mAP",
       title="Bandwidth efficiency: mAP vs total frames used")
ax.legend()
ax.grid(True, alpha=0.3)
fig.tight_layout()
plt.show()

## 7 Comparison with streaming centralized results

Load the streaming (centralized) results on the same manifest for side-by-side comparison.

In [ ]:
# Load corresponding streaming runs for comparison
STREAM_RUNS: dict[str, Path] = _runs_non_null(
    {
        # City-day bootstrap
        "no_filter_cityday_road_type": pick("streaming", "no_filter_cityday_road_type"),
        "random_filter_cityday_road_type": pick("streaming", "random_filter_cityday_road_type"),
        "dist_thresh_cityday_road_type_p10": pick("streaming", "dist_thresh_cityday_road_type_p10"),
        "dist_thresh_cityday_reverse_p10": pick("streaming", "dist_thresh_cityday_reverse_p10"),
        # City-mixed bootstrap
        "no_filter_citymix_road_type": pick("streaming", "no_filter_citymix_road_type"),
        "random_filter_citymix_road_type": pick("streaming", "random_filter_citymix_road_type"),
        "dist_thresh_citymix_road_type_p10": pick("streaming", "dist_thresh_citymix_road_type_p10"),
        "dist_thresh_citymix_reverse_p10": pick("streaming", "dist_thresh_citymix_reverse_p10"),
        "dist_thresh_citymix_conditions_p10": pick("streaming", "dist_thresh_citymix_conditions_p10"),
    }
)

STREAM_CK: dict[str, pd.DataFrame] = {}
for k, p in STREAM_RUNS.items():
    ck = ah.read_csv(p / "checkpoints.csv")
    if ck is not None and not ck.empty:
        STREAM_CK[k] = ck
        print(f"Streaming: {k:45s}  {len(ck)} checkpoints")

# Compare final mAP
print("\n--- Final mAP comparison ---")
print(f"{'Experiment':<55s} {'mAP':>8s}  {'mAP@50':>8s}  {'AP_Ped':>8s}")
print("-" * 85)

for k, rd in sorted(ROUNDS.items()):
    last = rd[rd["mAP"].notna()].iloc[-1] if "mAP" in rd.columns and rd["mAP"].notna().any() else None
    if last is not None:
        ped = float(last.get("AP_Pedestrian", 0)) if "AP_Pedestrian" in rd.columns else 0
        print(f"[Fed]  {SHORT_NAMES.get(k, k):<50s} {float(last['mAP']):8.4f}  "
              f"{float(last['mAP_50']):8.4f}  {ped:8.4f}")

for k, ck in sorted(STREAM_CK.items()):
    if "mAP" in ck.columns and ck["mAP"].notna().any():
        last = ck[ck["mAP"].notna()].iloc[-1]
        ped = float(last.get("AP_Pedestrian", 0)) if "AP_Pedestrian" in ck.columns else 0
        print(f"[Str]  {k:<50s} {float(last['mAP']):8.4f}  "
              f"{float(last['mAP_50']):8.4f}  {ped:8.4f}")

## 8 Summary table

In [ ]:
rows = []
for k, rd in sorted(ROUNDS.items()):
    cfg = ah.load_run_config(RUN[k])
    n_clients = int(cfg.get("num_clients", 4))

    total_accepted = 0
    total_items = 0
    for cid in range(n_clients):
        acc_col = f"client_{cid}_accepted"
        items_col = f"client_{cid}_items"
        if acc_col in rd.columns:
            total_accepted += int(rd[acc_col].astype(float).sum())
        if items_col in rd.columns:
            total_items += int(rd[items_col].astype(float).sum())

    last_mAP = 0.0
    last_mAP50 = 0.0
    last_ped = 0.0
    best_mAP = 0.0
    if "mAP" in rd.columns and rd["mAP"].notna().any():
        valid = rd[rd["mAP"].notna()]
        last_mAP = float(valid["mAP"].iloc[-1])
        last_mAP50 = float(valid["mAP_50"].iloc[-1])
        best_mAP = float(valid["mAP"].max())
        if "AP_Pedestrian" in valid.columns:
            last_ped = float(valid["AP_Pedestrian"].iloc[-1])

    accept_rate = total_accepted / total_items if total_items > 0 else 0
    rows.append({
        "Experiment": SHORT_NAMES.get(k, k),
        "Filter": str(cfg.get("filter_policy", "?")),
        "Clients": n_clients,
        "Rounds": len(rd),
        "Total items": total_items,
        "Accepted": total_accepted,
        "Accept %": f"{accept_rate:.1%}",
        "Final mAP": f"{last_mAP:.4f}",
        "Best mAP": f"{best_mAP:.4f}",
        "Final mAP@50": f"{last_mAP50:.4f}",
        "Final AP_Ped": f"{last_ped:.4f}",
    })

summary_df = pd.DataFrame(rows)
summary_df